# GAT Baseline

# GAT Baseline - Local Version

In [1]:
from pathlib import Path

# VARIAVEIS DE CONTROLE
NUM_NODES = 500 # Filtra os Top K genes mais variáveis
CONFIDENCE_THRESHOLD = 500 # Limiar de confiança entre as interações (0-1000).
PROCESSED_DATASET_PATH = Path('./data/processed/') # Caminho para o dataset já processado. Caso o diretorio não exista, o processamento do dataset será ativado.
BATCH_SIZE = 32
CHECKPOINT_INTERVAL = 500  # Define a cada quantas épocas um checkpoint será salvo
NUM_EPOCHS = 5000
CHECKPOINT_PATH = None # Carrega um modelo já pré-treinado. Adicione uma lista no formato ["diretorio da pasta", "nome do arquivo"], por exemplo: ['./data/processed/run_20260422_114036', 'checkpoint_epoch_2500.pt']
STOPPED_EPOCH = 2500

In [2]:
import os
import pandas as pd
import numpy as np
import torch
import requests
import io
from torch_geometric.data import Data

if PROCESSED_DATASET_PATH and PROCESSED_DATASET_PATH.joinpath(f'tcga_pacientes_{NUM_NODES}_{CONFIDENCE_THRESHOLD}.pt').exists():
    print(f"Carregando dataset processado: {PROCESSED_DATASET_PATH.joinpath(f'tcga_pacientes_{NUM_NODES}_{CONFIDENCE_THRESHOLD}.pt')}")
    dataset_pacientes = torch.load(PROCESSED_DATASET_PATH.joinpath(f'tcga_pacientes_{NUM_NODES}_{CONFIDENCE_THRESHOLD}.pt'), weights_only=False)
else:
    print("Iniciando processamento do zero (URLs Oficiais)...")

    # URLs Xena Browser
    url_exp = "https://gdc-hub.s3.us-east-1.amazonaws.com/download/TCGA-SKCM.star_tpm.tsv.gz"
    url_map = "https://gdc-hub.s3.us-east-1.amazonaws.com/download/gencode.v36.annotation.gtf.gene.probemap"

    # Download e Mapeamento
    df_exp = pd.read_csv(url_exp, sep='\t', index_col=0)
    df_map = pd.read_csv(url_map, sep='\t', usecols=['id', 'gene']).set_index('id')
    df_exp = df_exp.join(df_map).dropna(subset=['gene']).set_index('gene')
    df_exp = df_exp[~df_exp.index.duplicated(keep='first')]

    # Seleção de Genes (Variância)
    top_genes = df_exp.var(axis=1).nlargest(NUM_NODES).index
    df_filtered = df_exp.loc[top_genes]
    genes_list = df_filtered.index.tolist()

    # Rede PPI (STRING API)
    print("Consultando STRING PPI para os top genes...")
    r = requests.post("https://string-db.org/api/tsv/network",
                      data={"identifiers": "\r".join(genes_list), "species": 9606,
                            "required_score": CONFIDENCE_THRESHOLD, "network_type": "physical"})
    df_ppi = pd.read_csv(io.StringIO(r.text), sep='\t')

    gene_to_idx = {g: i for i, g in enumerate(genes_list)}
    edges = [[gene_to_idx[a], gene_to_idx[b]] for a, b in zip(df_ppi['preferredName_A'], df_ppi['preferredName_B'])
             if a in gene_to_idx and b in gene_to_idx]
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

    # Criação dos Grafos por Paciente (Usando o TCGA barcode)
    print("Construindo grafos individuais e extraindo rótulos...")
    dataset_pacientes = []
    for paciente_id in df_filtered.columns:
        barcode_parts = paciente_id.split('-')
        if len(barcode_parts) >= 4:
            sample_type = barcode_parts[3][:2]
            if sample_type == '01': y = 0 # Primário
            elif sample_type == '06': y = 1 # Metástase
            elif sample_type == '11': y = 2 # Normal / Sem doença
            else: continue

            x = torch.tensor(df_filtered[paciente_id].values, dtype=torch.float)
            x = torch.log1p(x).unsqueeze(1)
            data = Data(x=x, edge_index=edge_index, y=torch.tensor([y], dtype=torch.long))
            data.paciente_id = paciente_id
            data.gene_names = genes_list
            dataset_pacientes.append(data)

    print(f"Salvando {len(dataset_pacientes)} grafos em: {PROCESSED_DATASET_PATH.joinpath(f'tcga_pacientes_{NUM_NODES}_{CONFIDENCE_THRESHOLD}.pt')}")
    PROCESSED_DATASET_PATH.mkdir(parents=True, exist_ok=True)
    torch.save(dataset_pacientes, PROCESSED_DATASET_PATH.joinpath(f'tcga_pacientes_{NUM_NODES}_{CONFIDENCE_THRESHOLD}.pt'))

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Iniciando processamento do zero (URLs Oficiais)...


URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)>

import os
import pandas as pd
import numpy as np
import torch
import requests
import io
from torch_geometric.data import Data

if PROCESSED_DATASET_PATH and PROCESSED_DATASET_PATH.joinpath(f'tcga_pacientes_{NUM_NODES}_{CONFIDENCE_THRESHOLD}.pt').exists():
    print(f"Carregando dataset processado: {PROCESSED_DATASET_PATH.joinpath(f'tcga_pacientes_{NUM_NODES}_{CONFIDENCE_THRESHOLD}.pt')}")
    dataset_pacientes = torch.load(PROCESSED_DATASET_PATH.joinpath(f'tcga_pacientes_{NUM_NODES}_{CONFIDENCE_THRESHOLD}.pt'), weights_only=False)
else:
    print("Iniciando processamento do zero (URLs Oficiais)...")

    # URLs Xena Browser
    url_exp = "https://gdc-hub.s3.us-east-1.amazonaws.com/download/TCGA-SKCM.star_tpm.tsv.gz"
    url_map = "https://gdc-hub.s3.us-east-1.amazonaws.com/download/gencode.v36.annotation.gtf.gene.probemap"

    # Download e Mapeamento
    print("Baixando dados de expressão gênica do TCGA...")
    df_exp = pd.read_csv(url_exp, sep='\t', index_col=0)
    df_map = pd.read_csv(url_map, sep='\t', usecols=['id', 'gene']).set_index('id')
    df_exp = df_exp.join(df_map).dropna(subset=['gene']).set_index('gene')
    df_exp = df_exp[~df_exp.index.duplicated(keep='first')]

    # Seleção de Genes (Variância)
    print(f"Selecionando top {NUM_NODES} genes mais variáveis...")
    top_genes = df_exp.var(axis=1).nlargest(NUM_NODES).index
    df_filtered = df_exp.loc[top_genes]
    genes_list = df_filtered.index.tolist()

    # Rede PPI (STRING API)
    print("Consultando STRING PPI para os top genes...")
    r = requests.post("https://string-db.org/api/tsv/network",
                      data={"identifiers": "\r".join(genes_list), "species": 9606,
                            "required_score": CONFIDENCE_THRESHOLD, "network_type": "physical"})
    df_ppi = pd.read_csv(io.StringIO(r.text), sep='\t')

    gene_to_idx = {g: i for i, g in enumerate(genes_list)}
    edges = [[gene_to_idx[a], gene_to_idx[b]] for a, b in zip(df_ppi['preferredName_A'], df_ppi['preferredName_B'])
             if a in gene_to_idx and b in gene_to_idx]
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

    # Criação dos Grafos por Paciente (Usando o TCGA barcode)
    print("Construindo grafos individuais e extraindo rótulos...")
    dataset_pacientes = []
    for paciente_id in df_filtered.columns:
        barcode_parts = paciente_id.split('-')
        if len(barcode_parts) >= 4:
            sample_type = barcode_parts[3][:2]
            if sample_type == '01': y = 0 # Primário
            elif sample_type == '06': y = 1 # Metástase
            elif sample_type == '11': y = 2 # Normal / Sem doença
            else: continue

            x = torch.tensor(df_filtered[paciente_id].values, dtype=torch.float)
            x = torch.log1p(x).unsqueeze(1)
            data = Data(x=x, edge_index=edge_index, y=torch.tensor([y], dtype=torch.long))
            data.paciente_id = paciente_id
            data.gene_names = genes_list
            dataset_pacientes.append(data)

    print(f"Salvando {len(dataset_pacientes)} grafos em: {PROCESSED_DATASET_PATH.joinpath(f'tcga_pacientes_{NUM_NODES}_{CONFIDENCE_THRESHOLD}.pt')}")
    PROCESSED_DATASET_PATH.mkdir(parents=True, exist_ok=True)
    torch.save(dataset_pacientes, PROCESSED_DATASET_PATH.joinpath(f'tcga_pacientes_{NUM_NODES}_{CONFIDENCE_THRESHOLD}.pt'))

print(f"Dataset carregado: {len(dataset_pacientes)} pacientes")

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv, global_mean_pool
from torch_geometric.nn import LayerNorm

class GATv2Classifier(torch.nn.Module):
    def __init__(self, hidden_channels=64, num_classes=3):
        super().__init__()
        self.conv1 = GATv2Conv(1, hidden_channels, heads=4)
        self.norm1 = LayerNorm(hidden_channels * 4)
        self.conv2 = GATv2Conv(hidden_channels * 4, hidden_channels, heads=1)
        self.norm2 = LayerNorm(hidden_channels)
        self.lin = torch.nn.Linear(hidden_channels, num_classes)

    def forward(self, x, edge_index, batch, return_attn=False):
        x, (edge_index_attn, alpha) = self.conv1(x, edge_index, return_attention_weights=True)
        x = self.norm1(x)
        x = F.elu(x)
        x = F.dropout(x, p=0.4, training=self.training)

        x, _ = self.conv2(x, edge_index, return_attention_weights=True)
        x = self.norm2(x)
        x = F.elu(x)

        x = global_mean_pool(x, batch)
        x = self.lin(x)

        if return_attn: return x, edge_index_attn, alpha
        return x

## Treinamento

In [ ]:
from torch_geometric.loader import DataLoader
from datetime import datetime
from collections import Counter
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GATv2Classifier(hidden_channels=64, num_classes=3).to(device)
if CHECKPOINT_PATH:
  RUN_DIR = Path(CHECKPOINT_PATH[0])
  estado_salvo = torch.load(RUN_DIR.joinpath(CHECKPOINT_PATH[1]), map_location=device, weights_only=True)
  model.load_state_dict(estado_salvo)
  print(f"Pesos recuperados com sucesso do checkpoint: {RUN_DIR.joinpath(CHECKPOINT_PATH[1]).name}")
else:
  # Cria o diretório da execução usando pathlib
  timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
  RUN_DIR = PROCESSED_DATASET_PATH.joinpath(f"run_{timestamp}")
  RUN_DIR.mkdir(parents=True, exist_ok=True)

print(f"Os checkpoints e o modelo final serão salvos em: {RUN_DIR.name}")

train_loader = DataLoader(dataset_pacientes, batch_size=BATCH_SIZE, shuffle=True)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Conta a frequência de cada classe no dataset
labels = [data.y.item() for data in dataset_pacientes]
count = Counter(labels)

# Garante que as classes estão na ordem correta [0, 1, 2]
class_counts = [count[0], count[1], count[2]]
total_samples = sum(class_counts)

# Calcula os pesos: total / (n_classes * count)
# Damos um peso maior para as classes com menos exemplos
weights = [total_samples / (3.0 * c) if c > 0 else 0 for c in class_counts]
class_weights = torch.FloatTensor(weights).to(device)

print(f"Distribuição das classes: Primário: {class_counts[0]}, Metástase: {class_counts[1]}, Normal: {class_counts[2]}")
print(f"Pesos calculados para a Loss: {weights}")

# Atualiza a Função de Perda com os pesos
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

print(f"\nTreinando em {device}...")
model.train()
for epoch in range(STOPPED_EPOCH+1 if STOPPED_EPOCH else 1, NUM_EPOCHS+1):
    total_loss = 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data.x, data.edge_index, data.batch)
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Log de progresso
    if epoch % 10 == 0:
        print(f"Epoch {epoch:02d} | Loss: {total_loss/len(train_loader):.4f}")

    # Salva os Checkpoints (apenas os pesos da rede)
    if epoch % CHECKPOINT_INTERVAL == 0:
        ckpt_path = RUN_DIR.joinpath(f"checkpoint_epoch_{epoch:02d}.pt")
        torch.save(model.state_dict(), ckpt_path)

torch.save(model.state_dict(), RUN_DIR.joinpath("modelo_final.pt"))

print(f"\nTreinamento finalizado.")
print(f"Modelo final para inferência salvo em: {RUN_DIR.joinpath("modelo_final.pt")}")

## Visualização

## Visualização

Para rodar a aplicação de visualização:
```bash
streamlit run app.py
```

O app estará disponível em http://localhost:8501

In [ ]:
import os
import time
from pyngrok import ngrok

# 1. Matar qualquer processo travado
!pkill -f streamlit
ngrok.kill() # Limpa túneis antigos

# 2. Configurar Autenticação
ngrok.set_auth_token(NGROK_TOKEN)

# 3. Iniciar o Streamlit (agora de forma nativa e limpa)
print("Iniciando o servidor Streamlit...")
os.system("streamlit run app.py &> logs.txt &")
time.sleep(3) # Pausa para garantir que o servidor subiu

# 4. Criar o Túnel Profissional
tunnel = ngrok.connect(8501)
print(f"Dashboard ONLINE e estável: {tunnel.public_url}")

In [ ]:
%%writefile app.py
import streamlit as st
import torch
import networkx as nx
import plotly.graph_objects as go
import numpy as np
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv, global_mean_pool
from torch_geometric.nn import LayerNorm

class GATv2Classifier(torch.nn.Module):
    def __init__(self, hidden_channels=64, num_classes=3):
        super().__init__()
        self.conv1 = GATv2Conv(1, hidden_channels, heads=4)
        self.norm1 = LayerNorm(hidden_channels * 4)
        self.conv2 = GATv2Conv(hidden_channels * 4, hidden_channels, heads=1)
        self.norm2 = LayerNorm(hidden_channels)
        self.lin = torch.nn.Linear(hidden_channels, num_classes)

    def forward(self, x, edge_index, batch, return_attn=False):
        x, (edge_index_attn, alpha) = self.conv1(x, edge_index, return_attention_weights=True)
        x = self.norm1(x)
        x = F.elu(x)
        x = F.dropout(x, p=0.4, training=self.training)

        x, _ = self.conv2(x, edge_index, return_attention_weights=True)
        x = self.norm2(x)
        x = F.elu(x)

        x = global_mean_pool(x, batch)
        x = self.lin(x)

        if return_attn: return x, edge_index_attn, alpha
        return x

@st.cache_resource
def load_data(path):
    return torch.load(path, weights_only=False)

@st.cache_resource
def load_model(path, device):
    model = GATv2Classifier(hidden_channels=64, num_classes=3)
    model.load_state_dict(torch.load(path, map_location=device))
    model.eval()
    return model

# --- CONFIGURAÇÃO DA PÁGINA ---
st.set_page_config(page_title="BioGNN Explorer", layout="wide")
st.title("Visualizador de Atenção GATv2")

# Sidebar de Configurações
st.sidebar.header("Configurações de Dados")
data_path = st.sidebar.text_input("Caminho do Dataset (.pt)", "./data/processed/tcga_pacientes_500_500.pt")
model_path = st.sidebar.text_input("Caminho do Modelo (.pt)", "./data/processed/run_.../modelo_final.pt")

if st.sidebar.button("Carregar Dados e Modelo"):
    st.session_state['data_loaded'] = True

if st.session_state.get('data_loaded'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    dataset = load_data(data_path)
    model = load_model(model_path, device)

    # Controles da Interface
    pacientes_ids = [d.paciente_id for d in dataset]
    selected_id = st.selectbox("Selecione o Paciente para Análise:", pacientes_ids)

    vis_mode = st.radio(
        "Modo de Visualização do Grafo:",
        ["Subgrafo (Ocultar arestas irrelevantes)", "Grafo Completo (Destacar atenção por cor)"],
        horizontal=True
    )

    threshold = st.slider("Limiar de Atenção (Threshold)", 0.0, 1.0, 0.05, 0.01)

    # Inferência
    data = dataset[pacientes_ids.index(selected_id)].to(device)
    with torch.no_grad():
        out, edge_idx_attn, alpha = model(
            data.x, data.edge_index,
            torch.zeros(data.x.size(0), dtype=torch.long, device=device),
            return_attn=True
        )

        prob = torch.softmax(out, dim=1).cpu().numpy()[0]
        alpha_mean = alpha.mean(dim=1).cpu().numpy()
        edge_idx_attn = edge_idx_attn.cpu().numpy()

    # Layout em colunas
    col1, col2 = st.columns([1, 3])
    with col1:
        classes_map = {0: "Tumor Primário", 1: "Metástase", 2: "Tecido Saudável"}
        pred_idx = np.argmax(prob)

        st.metric("Classe Predita", classes_map[pred_idx])
        st.write(f"**Confiança Absoluta:** {prob[pred_idx]*100:.2f}%")

        st.write("---")
        st.write("**Distribuição das Probabilidades:**")
        st.progress(float(prob[0]), text=f"Primário ({prob[0]*100:.1f}%)")
        st.progress(float(prob[1]), text=f"Metástase ({prob[1]*100:.1f}%)")
        st.progress(float(prob[2]), text=f"Saudável ({prob[2]*100:.1f}%)")
        st.write("---")

        # Legenda do Gradiente
        st.write("**Mapa de Cores da Atenção (Arestas):**")
        st.markdown(
            "Limiar (" + str(threshold) + ") <span style='color:rgb(255,180,0)'>■</span> "
            "➔ <span style='color:rgb(217,90,0)'>■</span> "
            "➔ <span style='color:rgb(180,0,0)'>■</span> Máxima Atenção",
            unsafe_allow_html=True
        )

    # Construção do Grafo Visual
    with col2:
        G = nx.Graph()
        nomes_genes = getattr(data, 'gene_names', [f"Gene {idx}" for idx in range(data.x.size(0))])

        for i, gene in enumerate(nomes_genes):
            G.add_node(i, name=gene, expression=data.x[i].item())

        for i in range(edge_idx_attn.shape[1]):
            u, v = edge_idx_attn[0, i], edge_idx_attn[1, i]
            G.add_edge(u, v)

        pos = nx.spring_layout(G, k=0.2, seed=42)

        # Preparando os vetores para otimização
        low_x, low_y = [], []

        # Cria 10 "baldes" de cores para as conexões importantes
        num_bins = 10
        color_bins = {i: {"x": [], "y": []} for i in range(num_bins)}

        max_alpha = alpha_mean.max()

        # Distribuindo as arestas nos baldes
        for i in range(edge_idx_attn.shape[1]):
            u, v = edge_idx_attn[0, i], edge_idx_attn[1, i]
            w = alpha_mean[i]
            x0, y0 = pos[u]
            x1, y1 = pos[v]

            if w >= threshold:
                # Normaliza o peso entre 0 e 1 (sendo 0 o threshold e 1 o peso máximo)
                norm_w = (w - threshold) / (max_alpha - threshold + 1e-9)
                norm_w = max(0.0, min(1.0, norm_w)) # Garante que fique entre 0 e 1

                # Descobre em qual balde (0 a 9) a aresta cai
                bin_idx = min(num_bins - 1, int(norm_w * num_bins))
                color_bins[bin_idx]["x"].extend([x0, x1, None])
                color_bins[bin_idx]["y"].extend([y0, y1, None])
            else:
                if "Completo" in vis_mode:
                    low_x.extend([x0, x1, None])
                    low_y.extend([y0, y1, None])

        edge_traces = []

        # Desenha arestas de baixa atenção (fundo cinza)
        if low_x:
            edge_traces.append(go.Scatter(
                x=low_x, y=low_y,
                line=dict(width=0.5, color='rgba(150,150,150,0.15)'),
                hoverinfo='none', mode='lines'
            ))

        # Desenha as arestas de alta atenção com o Gradiente de Cores (Laranja -> Vermelho Escuro)
        for i in range(num_bins):
            if color_bins[i]["x"]:
                norm = i / (num_bins - 1) # Vai de 0.0 a 1.0

                # Interpolação de cor (De rgb(255,180,0) até rgb(180,0,0))
                r = int(255 - norm * 75)
                g = int(180 - norm * 180)
                b = 0

                # Interpolação de espessura (De 1.5 a 4.5 pixels)
                w_line = 1.5 + norm * 3.0

                edge_traces.append(go.Scatter(
                    x=color_bins[i]["x"], y=color_bins[i]["y"],
                    line=dict(width=w_line, color=f'rgba({r},{g},{b},0.9)'),
                    hoverinfo='none', mode='lines'
                ))

        # Plotly: Nós
        node_x, node_y, node_text, node_color = [], [], [], []
        for node in G.nodes():
            x, y = pos[node]
            node_x.append(x)
            node_y.append(y)
            node_text.append(f"Gene: {G.nodes[node]['name']}<br>Expressão: {G.nodes[node]['expression']:.2f}")
            node_color.append(G.nodes[node]['expression'])

        node_trace = go.Scatter(
            x=node_x, y=node_y, mode='markers', hoverinfo='text',
            text=node_text,
            marker=dict(showscale=True, colorscale='Viridis', size=10, color=node_color,
                        colorbar=dict(thickness=15, title='Nível de Expressão'))
        )

        fig = go.Figure(data=edge_traces + [node_trace],
                     layout=go.Layout(showlegend=False, hovermode='closest',
                                      margin=dict(b=0,l=0,r=0,t=0),
                                      xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                                      yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)))

        st.plotly_chart(fig, use_container_width=True)
else:
    st.warning("Aguardando carregamento dos dados...")